# Programação III — Ficha Prática 7
> **Recursos:** [Enunciado em PDF](./Sheet_7.pdf)

| Informação | Detalhe |
| :--- | :--- |
| **Estudante** | Gonçalo Pereira |
| **Número de Aluno** | 2022051 |
| **Curso / Ano** | Licenciatura em Engenharia Informática (LEI) — 2.º Ano |
| **Unidade Curricular**| Programação III |

---

> **Nota:** Por motivos de padronização técnica e boas práticas de desenvolvimento de software, os comentários, a documentação de código e os títulos das secções de cada exercício encontram-se redigidos em **Inglês**.

In [13]:
from abc import ABC, abstractmethod
from datetime import date
import uuid


class LibrarianItem(ABC):
    """Abstract base class representing a generic item in the library catalog.

    Attributes:
        id (str): Universally unique identifier generated automatically (read-only).
    """

    def __init__(self) -> None:
        """Initializes a new LibrarianItem with a unique UUID."""
        self.__id = str(uuid.uuid4())

    # -------------------------------------------------------------------------
    # Public & Abstract Methods
    # -------------------------------------------------------------------------
    def get_id(self) -> str:
        """Returns the unique identifier of the library item.

        Returns:
            str: UUID string identifying the item.
        """
        return self.__id

    @abstractmethod
    def get_desc(self) -> str:
        """Generates a formatted description of the item.

        Returns:
            str: Human-readable summary of the item's details and status.
        """
        pass

    @abstractmethod
    def can_borrow(self) -> bool:
        """Checks whether the item is currently eligible for borrowing.

        Returns:
            bool: True if the item can be borrowed, False otherwise.
        """
        pass


class Person(ABC):
    """Abstract base class representing a person interacting with the library.

    Attributes:
        _name (str): Protected name of the person.
    """

    def __init__(self, name: str) -> None:
        """Initializes a new Person instance.

        Args:
            name (str): Full name of the person.

        Raises:
            ValueError: If name is not a string or is empty.
        """
        if not isinstance(name, str) or not name.strip():
            raise ValueError("O nome não pode estar vazio.")
        self._name = name.strip()

    # -------------------------------------------------------------------------
    # Public & Abstract Methods
    # -------------------------------------------------------------------------
    @abstractmethod
    def validate_identity(self) -> bool:
        """Validates the person's credentials or account eligibility.

        Returns:
            bool: True if identity/status is valid, False otherwise.
        """
        pass

    def get_name(self) -> str:
        """Returns the person's name.

        Returns:
            str: Stored name of the person.
        """
        return self._name


class Book(LibrarianItem):
    """Concrete class representing a book in the library catalog.

    Attributes:
        isbn (str): International Standard Book Number (read-only).
        title (str): Title of the book.
        author (str): Author of the book.
        available (bool): Current availability status for borrowing.
        acquisition_date (date): Date when the book was acquired by the library.
    """

    def __init__(self, isbn: str, title: str, author: str, acquisition_date: date | None = None) -> None:
        """Initializes a new Book instance.

        Args:
            isbn (str): Unique ISBN identifier for the book edition.
            title (str): Title of the book.
            author (str): Author of the book.
            acquisition_date (date | None, optional): Date of acquisition. Defaults to today.

        Raises:
            ValueError: If isbn is not a non-empty string.
        """
        super().__init__()

        if not isinstance(isbn, str) or not isbn.strip():
            raise ValueError("ISBN must be a non-empty string.")
        self.__isbn = isbn.strip()

        self.__title = title
        self.__author = author

        self._available = True
        self._acquisition_date = acquisition_date if acquisition_date is not None else date.today()

    # -------------------------------------------------------------------------
    # Properties & Setters
    # -------------------------------------------------------------------------
    @property
    def isbn(self) -> str:
        return self.__isbn

    @property
    def title(self) -> str:
        return self.__title

    @title.setter
    def title(self, value: str) -> None:
        if not isinstance(value, str):
            raise TypeError("Title must be a string")
        if not value.strip():
            raise ValueError("Title cannot be empty")
        self.__title = value.strip()

    @property
    def author(self) -> str:
        return self.__author

    @author.setter
    def author(self, value: str) -> None:
        if not isinstance(value, str):
            raise TypeError("Author must be a string")
        if not value.strip():
            raise ValueError("Author cannot be empty")
        self.__author = value.strip()

    @property
    def available(self) -> bool:
        return self._available

    @available.setter
    def available(self, value: bool) -> None:
        if not isinstance(value, bool):
            raise TypeError("Available status must be a boolean (True or False).")
        self._available = value

    @property
    def acquisition_date(self) -> date:
        return self._acquisition_date

    @acquisition_date.setter
    def acquisition_date(self, value: date) -> None:
        if not isinstance(value, date):
            raise TypeError("Acquisition date must be a datetime.date object.")
        if value > date.today():
            raise ValueError("Acquisition date cannot be in the future.")
        self._acquisition_date = value

    # -------------------------------------------------------------------------
    # Overridden Abstract Methods
    # -------------------------------------------------------------------------
    def get_desc(self) -> str:
        """Returns a formatted summary of the book and its availability.

        Returns:
            str: Description containing title, author, ISBN, and availability status.
        """
        status = "Disponível" if self._available else "Indisponível"
        return f"Book: '{self.__title}' por {self.__author} (ISBN: {self.__isbn}) - [{status}]"

    def can_borrow(self) -> bool:
        """Checks whether the book is currently available for borrowing.

        Returns:
            bool: True if available, False otherwise.
        """
        return self._available


class Member(Person):
    """Concrete class representing a registered library member.

    Attributes:
        member_number (str): Unique membership identifier (read-only).
        credit_card (str): Masked credit card number showing only the last 4 digits.
    """

    def __init__(self, name: str, member_number: str, credit_card: str) -> None:
        """Initializes a new Member instance.

        Args:
            name (str): Full name of the member.
            member_number (str): Unique membership ID.
            credit_card (str): 16-digit credit card number on file.

        Raises:
            ValueError: If member_number is empty.
        """
        super().__init__(name)

        if not isinstance(member_number, str) or not member_number.strip():
            raise ValueError("O número de membro é obrigatório.")
        self.__member_number = member_number

        self.__credit_card = credit_card

        self._active_loans = []
        self._pending_fines = 0.0

    # -------------------------------------------------------------------------
    # Properties & Setters
    # -------------------------------------------------------------------------
    @property
    def member_number(self) -> str:
        return self.__member_number

    @property
    def credit_card(self) -> str:
        return f"****-****-****-{self.__credit_card[-4:]}"

    @credit_card.setter
    def credit_card(self, value: str) -> None:
        if not isinstance(value, str):
            raise TypeError("O número do cartão deve ser uma string.")
        digits = value.replace("-", "").replace(" ", "")
        if not digits.isdigit() or len(digits) != 16:
            raise ValueError("O cartão de crédito deve conter exatamente 16 dígitos.")
        self.__credit_card = digits

    # -------------------------------------------------------------------------
    # Public & Overridden Methods
    # -------------------------------------------------------------------------
    def renew_card(self, new_card: str, cvv: str) -> bool:
        """Updates the member's credit card after validating the security code.

        Args:
            new_card (str): New 16-digit credit card number.
            cvv (str): 3- or 4-digit card security code.

        Returns:
            bool: True if the card was successfully updated.

        Raises:
            ValueError: If cvv format is invalid or new_card does not have 16 digits.
        """
        if not isinstance(cvv, str) or not cvv.isdigit() or len(cvv) not in (3, 4):
            raise ValueError("CVV inválido. Deve conter 3 ou 4 dígitos.")
        self.credit_card = new_card
        return True

    def validate_identity(self) -> bool:
        """Validates if the member has a valid ID, valid card, and no pending fines.

        Returns:
            bool: True if the member is in good standing, False otherwise.
        """
        return bool(self.__member_number) and len(self.__credit_card) == 16 and self._pending_fines == 0.0


class Librarian(Person):
    """Concrete class representing a library staff member.

    Attributes:
        employee_number (str): Unique staff identifier (read-only).
        password (str): Account password (minimum 6 characters).
    """

    def __init__(self, name: str, employee_number: str, password: str) -> None:
        """Initializes a new Librarian instance.

        Args:
            name (str): Full name of the librarian.
            employee_number (str): Unique staff ID.
            password (str): Authentication password.

        Raises:
            ValueError: If employee_number is empty.
        """
        super().__init__(name)

        if not isinstance(employee_number, str) or not employee_number.strip():
            raise ValueError("O número de funcionário é obrigatório.")
        self.__employee_number = employee_number

        self.__password = password

    # -------------------------------------------------------------------------
    # Properties & Setters
    # -------------------------------------------------------------------------
    @property
    def employee_number(self) -> str:
        return self.__employee_number

    @property
    def password(self) -> str:
        return self.__password

    @password.setter
    def password(self, value: str) -> None:
        if not isinstance(value, str):
            raise TypeError("A senha deve ser uma string.")
        if len(value) < 6:
            raise ValueError("A senha deve ter pelo menos 6 caracteres.")
        self.__password = value

    # -------------------------------------------------------------------------
    # Protected & Overridden Methods
    # -------------------------------------------------------------------------
    def _validate_credentials(self, employee: str, password: str) -> bool:
        """Verifies whether the provided employee ID and password match stored records.

        Args:
            employee (str): Employee number to verify.
            password (str): Password to verify.

        Returns:
            bool: True if both credentials match, False otherwise.
        """
        if not isinstance(employee, str) or not isinstance(password, str):
            return False
        return (
            self.__employee_number == employee.strip()
            and self.__password == password
        )

    def validate_identity(self) -> bool:
        """Validates whether the librarian holds valid active credentials.

        Returns:
            bool: True if employee number is set and password meets length rules.
        """
        return bool(self.__employee_number) and len(self.__password) >= 6


# -----------------------------------------------------------------------------
# Test Execution & Library Simulation
# -----------------------------------------------------------------------------

# Instantiate book copies (same ISBN, unique auto-generated UUIDs)
print("1. CRIAR LIVROS")
book1 = Book("978-972-0-04957-5", "Os Maias", "Eça de Queirós", date(2022, 3, 15))
book2 = Book("978-972-0-04957-5", "Os Maias", "Eça de Queirós")
book3 = Book("978-972-20-3754-9", "Ensaio sobre a Cegueira", "José Saramago", date(2024, 1, 10))

print(f"[{book1.get_id()}] {book1.get_desc()} | Aquisição: {book1.acquisition_date}")
print(f"[{book2.get_id()}] {book2.get_desc()} | Aquisição: {book2.acquisition_date}")
print(f"[{book3.get_id()}] {book3.get_desc()} | Pode emprestar? {book3.can_borrow()}")

# Test updating book availability via property setter
book3.available = False
print(f"Após empréstimo -> {book3.get_desc()} | Pode emprestar? {book3.can_borrow()}")

# Instantiate library members and test card masking and renewal
print("\n2. CRIAR MEMBROS")
member1 = Member("Tiago Ferreira", "MEM-101", "4532015112830366")
member2 = Member("Beatriz Sousa", "MEM-102", "5425233430109903")

print(f"Membro 1: {member1.get_name()} | Nº: {member1.member_number} | Cartão: {member1.credit_card}")
print(f"Identidade válida? {member1.validate_identity()}")

# Renew credit card for Member 2
print(f"\nMembro 2: {member2.get_name()} | Cartão antigo: {member2.credit_card}")
member2.renew_card("4916384729104821", "842")
print(f"Cartão renovado: {member2.credit_card} | Identidade válida? {member2.validate_identity()}")

# Instantiate librarians and test credential verification
print("\n3. CRIAR BIBLIOTECÁRIOS")
librarian1 = Librarian("Dr.ª Helena Matos", "FUNC-01", "Porto2026!")
librarian2 = Librarian("Rui Carvalho", "FUNC-02", "Biblio#99")

print(f"Bibliotecária: {librarian1.get_name()} | Nº: {librarian1.employee_number}")
print(f"Identidade válida? {librarian1.validate_identity()}")
print("Credenciais válidas (certas):", librarian1._validate_credentials("FUNC-01", "Porto2026!"))
print("Credenciais válidas (erradas):", librarian2._validate_credentials("FUNC-02", "senha_errada"))

# Test property setters and expected validation exceptions
print("\n4. TESTAR VALIDAÇÕES DOS SETTERS (EXCEÇÕES ESPERADAS)")

# Attempt to assign an empty title (should fail)
try:
    book1.title = "   "
except ValueError as erro:
    print(f"[Bloqueado] {erro}")

# Attempt to renew card with invalid digits and CVV (should fail)
try:
    member1.renew_card("1234", "999")
except ValueError as erro:
    print(f"[Bloqueado] {erro}")

# Attempt to set a password shorter than 6 characters (should fail)
try:
    librarian2.password = "123"
except ValueError as erro:
    print(f"[Bloqueado] {erro}")

1. CRIAR LIVROS
[63e9dd7d-0625-46d5-b512-18b13f3141d3] Book: 'Os Maias' por Eça de Queirós (ISBN: 978-972-0-04957-5) - [Disponível] | Aquisição: 2022-03-15
[1e0b5f4b-dd8e-4486-88ae-89617b28f633] Book: 'Os Maias' por Eça de Queirós (ISBN: 978-972-0-04957-5) - [Disponível] | Aquisição: 2026-11-25
[c7ab8fb0-58cf-43f7-a992-9be2be801b0d] Book: 'Ensaio sobre a Cegueira' por José Saramago (ISBN: 978-972-20-3754-9) - [Disponível] | Pode emprestar? True
Após empréstimo -> Book: 'Ensaio sobre a Cegueira' por José Saramago (ISBN: 978-972-20-3754-9) - [Indisponível] | Pode emprestar? False

2. CRIAR MEMBROS
Membro 1: Tiago Ferreira | Nº: MEM-101 | Cartão: ****-****-****-0366
Identidade válida? True

Membro 2: Beatriz Sousa | Cartão antigo: ****-****-****-9903
Cartão renovado: ****-****-****-4821 | Identidade válida? True

3. CRIAR BIBLIOTECÁRIOS
Bibliotecária: Dr.ª Helena Matos | Nº: FUNC-01
Identidade válida? True
Credenciais válidas (certas): True
Credenciais válidas (erradas): False

4. TESTAR 